# Data Loading, Storage, and File Formats

In [1]:
import pandas as pd

In [2]:
# Read from a CSV file (comma-separated values). Default: delimiter=','
pd.read_csv('examples/file.csv')

# Read from a text file with tab-separated values. Default: delimiter='\t'
pd.read_table('examples/file.txt')

# Read data in fixed-width column format (no delimiters)
pd.read_fwf('examples/file.txt')

# Read data from the clipboard (useful for copying tables from the web)
pd.read_clipboard(sep='\t')

,Health
0,Health
1,Connect and explore your health data
2,Open COROS
3,COROS
4,COROS
5,Workout data insights
6,Open Calorie Tracker
7,Calorie Tracker
8,Calorie Tracker
9,Track your food and calories


In [3]:
df1 = pd.read_csv('examples/file.csv', header=None)
# Columns will be named: 0, 1, 2, 3, 4

df2 = pd.read_csv('examples/file.csv', names=['a', 'b', 'c', 'd', 'message'])
# Columns will be named: a, b, c, d, message

In [4]:
# Use a regular expression '\s+' to split on any amount of whitespace (spaces, tabs)
result = pd.read_table('examples/file.txt', sep='\s+')
print(result)
# Note: Because there were fewer column names than data columns,
# pandas inferred the first data column as the index.

            A         B       C
aaa -0.264438 -1.026059 -0.6195


In [5]:
# Skip rows at positions 0, 2, and 3 (0-based indexing)
df_clean = pd.read_csv('examples/ex1.csv', skiprows=[0, 2, 3])
print(df_clean)
# Only the rows containing actual data are read.

Empty DataFrame
Columns: [a, b, c, d, message]
Index: []


In [6]:
# something,a,b,c,d,message
# one,1,2,3,4,NA
# two,5,6,,8,world  # <- سلول خالی بین 6 و 8
# three,9,10,11,12,foo

# By default, pandas recognizes common NA markers like 'NA', 'NULL', empty strings
result_default = pd.read_csv('examples/ex2.csv')
print(result_default)
# Empty cells and the value 'NA' become 'NaN'.

# Specify a custom list of strings to be treated as missing values
result_custom_na = pd.read_csv('examples/ex2.csv', na_values=['NULL'], keep_default_na=False)
print(f'\n{result_custom_na}')
# Only 'NULL' is recognized as missing (not 'NA').

# Specify different NA markers for each column using a dictionary
sentinels = {'message': ['foo', 'NA'], 'something': ['two']}
result_per_column = pd.read_csv('examples/ex2.csv', na_values=sentinels)
print(f'\n{result_per_column}')
# In 'message' column, 'foo' and 'NA' become NaN. In 'something' column, 'two' becomes NaN.

  something  a   b     c   d message
0       one  1   2   3.0   4     NaN
1       two  5   6   NaN   8   world
2     three  9  10  11.0  12     foo

  something  a   b   c   d message
0       one  1   2   3   4      NA
1       two  5   6       8   world
2     three  9  10  11  12     foo

  something  a   b     c   d message
0       one  1   2   3.0   4     NaN
1       NaN  5   6   NaN   8   world
2     three  9  10  11.0  12     NaN


### **Summary of Key Arguments for read_csv/read_table**

|Argument|	Description|	Common Example|
|--------|-------------|------------------|
sep / delimiter	|Field separator. Can be a character or a regular expression (Regex).	|`sep=',', sep='\t', sep='\s+'`
header|	Row number to use as column names. Use None if the file has no header.	|`header=0 (default), header=None`
names	|List of column names to use for the DataFrame. Often combined with `header=None`.	|`names=['id', 'name', 'value']`
index_col|	Column number or name to use as the DataFrame index. Use a list for a MultiIndex.	|`index_col=0, index_col='date', index_col=['city', 'year']`
skiprows|	Number of rows or a list of row indices to skip from the start of the file.	|`skiprows=3, skiprows=[0, 2, 5]`
na_values|	A list or dictionary of values to interpret as missing (NaN).|	`na_values=['NA', '--', '']`
comment	|Character indicating the start of a comment. The rest of the line is ignored.|	`comment='#'`
parse_dates|	Attempt to parse dates. Can be True (all columns), a list of columns, or a list of lists (to combine columns).|	`parse_dates=True, parse_dates=['birth_date'], parse_dates=[[1,2,3]]`
encoding	|Text encoding for the file. For Persian text, it's often `'utf-8'`.|	`encoding='utf-8', encoding='cp1256'` (older Windows Persian)
nrows	|Read only a limited number of rows from the start of the file (useful for initial sampling).	|`nrows=1000`
chunksize|	Read the file in chunks of this size. Crucial for processing very large files that don't fit in memory.	|`chunksize=10000`

#### **Final Notes and Summary**

- read_csv is the king: You will use this function 90% of the time.
- The main challenge is real-world data messiness: Parameters like `skiprows`, `comment`, `na_values`, and `encoding` are your tools to handle this mess.
- Always sample your data first: Before reading a huge file, use `nrows=1000` to inspect the file's structure and find the correct parameters.
- Take encoding seriously: If you see strange characters instead of text, the problem is likely the encoding. `'utf-8'` is usually the first guess.
- You are now ready to read almost any structured text file into pandas. In the next section of the book, you will learn how to write data to files (`to_csv`) and work with other formats like Excel and JSON.

#### **Pandas Defaults:**

- Pandas has a built-in list of values it recognizes as missing, which includes `NA`,` N/A`, `null`, `NULL`, `NaN`, `nan`, and empty cells.
- `keep_default_na`: When you specify custom na_values, the `keep_default_na` parameter determines whether the default list is also applied (default: `True`).

### Reading Text Files in Pieces

In [7]:
pd.options.display.max_rows = 10
# The first 10 lines and the last 10 lines

result = pd.read_csv('examples/ex3.csv')
result

,Car Make,Car Model,Year,Engine Size (L),Horsepower,Torque (lb-ft),0-60 MPH Time (seconds),Price (in USD)
0,Porsche,911,2022,3,379,331,4.0,"101,200"
1,Lamborghini,Huracan,2021,5.2,630,443,2.8,"274,390"
2,Ferrari,488 GTB,2022,3.9,661,561,3.0,"333,750"
3,Audi,R8,2022,5.2,562,406,3.2,"142,700"
4,McLaren,720S,2021,4,710,568,2.7,"298,000"
...,...,...,...,...,...,...,...,...
44,Porsche,Cayman,2021,2,300,280,5.1,"58,900"
45,Lamborghini,Aventador SVJ,2021,6.5,759,531,2.8,"518,000"
46,Ferrari,SF90 Stradale,2021,4,986,590,2.5,"625,000"
47,Audi,RS3,2022,2.5,394,369,3.9,"56,200"


In [8]:
pd.read_csv('examples/ex3.csv', nrows=5)
# The first 5 lines
chunker = pd.read_csv('examples/ex3.csv', chunksize=100)
# Output: <pandas.io.parsers.readers.TextFileReader at 0x76c6d8d1bbb0>


tot = pd.Series([])

for piece in chunker:
    tot = tot.add(piece['Year'].value_counts(), fill_value=0)

tot = tot.sort_values(ascending=False)


In [9]:
# Create a TextFileReader object with chunksize of 1000
chunker = pd.read_csv('examples/ex3.csv', chunksize=20)

# Read the first chunk
chunk1 = chunker.get_chunk(20)  # or next(chunker)

# Read the second chunk
chunk2 = chunker.get_chunk(20)

# Suppose the file only had 2000 rows total
# Now the iterator is exhausted

# This will raise StopIteration because there's no more data
# chunk3 = chunker.get_chunk(500)  ERROR: StopIteration!

### Writing Data to Text Format

In [10]:
# Read sample data from a CSV file
data = pd.read_csv('examples/file.csv')

# Write the DataFrame to a new CSV file
data.to_csv('examples/out.csv')

# Display the saved file content
with open('examples/out.csv', 'r') as f:
    print("\nSaved file content:")
    print(f.read())


Saved file content:
,1,2,3,4,hello
0,5,6,7,8,world
1,9,10,11,12,foo



In [11]:
import sys

# Write to console with pipe delimiter instead of comma
data.to_csv(sys.stdout, sep='|')

# Write with tab delimiter (TSV format)
data.to_csv('examples/out.tsv', sep='\t', index=False)

|1|2|3|4|hello
0|5|6|7|8|world
1|9|10|11|12|foo


In [12]:
# Replace NaN values with a custom string like 'NULL' or 'MISSING'
data.to_csv(sys.stdout, na_rep='NULL')

# You can use any string you want
data.to_csv('examples/out_na.csv', na_rep='MISSING_DATA')

,1,2,3,4,hello
0,5,6,7,8,world
1,9,10,11,12,foo


In [13]:
# Disable both index and header (clean data only)
data.to_csv(sys.stdout, index=False, header=False)

data.columns.tolist()

5,6,7,8,world
9,10,11,12,foo


['1', '2', '3', '4', 'hello']

In [14]:
# Write only specific columns in a custom order
data.to_csv(sys.stdout, index=False, columns=['1', '2', '3', '4', 'hello'])

1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo


In [15]:
import numpy as np

# Create a time series
dates = pd.date_range('1/1/2000', periods=7)
ts = pd.Series(np.arange(7), index=dates)

# Save the Series to CSV
ts.to_csv('examples/tseries.csv')

print("Series saved. Content:")
with open('examples/tseries.csv', 'r') as f:
    print(f.read())
# Output: 2000-01-01,0
#         2000-01-02,1
#         ...
#         2000-01-07,6

Series saved. Content:
,0
2000-01-01,0
2000-01-02,1
2000-01-03,2
2000-01-04,3
2000-01-05,4
2000-01-06,5
2000-01-07,6



In [16]:
import csv

# Open and read a CSV file using Python's csv module
with open('examples/file.csv') as f:
    reader = csv.reader(f)
    
    # Iterate through each row in the CSV file
    for line in reader:
        print(line)

['1', '2', '3', '4', 'hello']
['5', '6', '7', '8', 'world']
['9', '10', '11', '12', 'foo']


In [17]:
# Manual processing of CSV data
with open('examples/file.csv') as f:
    lines = list(csv.reader(f))

# Separate header from data
header, values = lines[0], lines[1:]
print(f"Header: {header}")
print(f"Values: {values}")

# Create a dictionary using dictionary comprehension and zip
data_dict = {h: v for h, v in zip(header, zip(*values))}
print(f"Data Dictionary: {data_dict}")

Header: ['1', '2', '3', '4', 'hello']
Values: [['5', '6', '7', '8', 'world'], ['9', '10', '11', '12', 'foo']]
Data Dictionary: {'1': ('5', '9'), '2': ('6', '10'), '3': ('7', '11'), '4': ('8', '12'), 'hello': ('world', 'foo')}


In [18]:
# Define a custom CSV dialect (format)
class my_dialect(csv.Dialect):
    lineterminator = '\n'      # Line ending character
    delimiter = ';'            # Field separator
    quotechar = '"'            # Quote character
    quoting = csv.QUOTE_MINIMAL  # When to use quotes

# Use the custom dialect
with open('examples/file.csv') as f:
    reader = csv.reader(f, dialect=my_dialect)
    for row in reader:
        print(row)

# Or use parameters directly without defining a class
with open('examples/file.csv') as f:
    reader = csv.reader(f, delimiter='|', quotechar='"')
    for row in reader:
        print(row)

['1,2,3,4,hello']
['5,6,7,8,world']
['9,10,11,12,foo']
['1,2,3,4,hello']
['5,6,7,8,world']
['9,10,11,12,foo']


In [19]:
# Write data using csv.writer
with open('examples/mydata.csv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter=',')
    
    # Write header row
    writer.writerow(['Name', 'Age', 'City'])
    
    # Write data rows
    writer.writerow(['Alice', 30, 'New York'])
    writer.writerow(['Bob', 25, 'London'])
    writer.writerow(['Charlie', 35, 'Tokyo'])

print("File 'mydata.csv' created successfully.")

File 'mydata.csv' created successfully.


### **CSV Dialect**
|Argument |	Description |	Common Values|
|---------|-------------|----------------|
delimiter	|One-character string to separate fields	|`','  ';' '\t'    '|'`
lineterminator|	Line terminator for writing |	`'\n', '\r\n'`
quotechar	|Character for quoting fields with special chars	| `'"', "'"`
quoting	|When to use quoting |	`csv.QUOTE_ALL, csv.QUOTE_MINIMAL, csv.QUOTE_NONNUMERIC, csv.QUOTE_NONE`
skipinitialspace|	Ignore whitespace after delimiter	|`True, False`
doublequote|	How to handle quotechar inside field	|`True, False`
escapechar|	Escape character if quoting is `QUOTE_NONE`	|`'\\', None`

- For persian note: `data.to_csv('output.csv', encoding='utf-8-sig')`
- Control missing data: `data.to_csv('output.csv', na_rep='NA'`
- For big data use compression : `data.to_csv('output.csv.gz', compression='gzip')`

### JSON Data

In [20]:
# Example JSON string
obj = """
{"name": "Wes",
"places_lived": ["United States", "Spain", "Germany"],
"pet": null,
"siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]},
{"name": "Katie", "age": 38,
"pets": ["Sixes", "Stache", "Cisco"]}]
}
"""

# Importing json module
import json

# Converting JSON string to Python object
result = json.loads(obj)

result

{'name': 'Wes',
 'places_lived': ['United States', 'Spain', 'Germany'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 30, 'pets': ['Zeus', 'Zuko']},
  {'name': 'Katie', 'age': 38, 'pets': ['Sixes', 'Stache', 'Cisco']}]}

In [21]:
# Converting Python object back to JSON
asjson = json.dumps(result)
asjson

'{"name": "Wes", "places_lived": ["United States", "Spain", "Germany"], "pet": null, "siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]}, {"name": "Katie", "age": 38, "pets": ["Sixes", "Stache", "Cisco"]}]}'

In [22]:
# Reading JSON file with pandas
data = pd.read_json('examples/ex1.json')
data

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


In [23]:
# Exporting DataFrame to JSON (default orientation)
print(data.to_json())
# Output:
# {"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}

# Exporting with 'records' orientation
print(data.to_json(orient='records'))
# Output:
# [{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]

{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}
[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]


In [24]:
# Different orientations for JSON output

# 'split' - Dict with columns, index, and data separated
print(data.to_json(orient='split'))

# 'records' - List of records (most common for APIs)
print(data.to_json(orient='records'))

# 'index' - Dict with index as keys
print(data.to_json(orient='index'))

# 'columns' - Dict with columns as keys
print(data.to_json(orient='columns'))

# 'values' - Just the values (2D array)
print(data.to_json(orient='values'))

{"columns":["a","b","c"],"index":[0,1,2],"data":[[1,2,3],[4,5,6],[7,8,9]]}
[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]
{"0":{"a":1,"b":2,"c":3},"1":{"a":4,"b":5,"c":6},"2":{"a":7,"b":8,"c":9}}
{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}
[[1,2,3],[4,5,6],[7,8,9]]


In [25]:
# Working with nested JSON data

# Accessing nested data
print(result['siblings'][0]['pets'][0])  # Output: 'Zeus'

# Flattening nested JSON (advanced technique)
siblings_df = pd.json_normalize(result['siblings'])
print(siblings_df)

Zeus
    name  age                    pets
0  Scott   30            [Zeus, Zuko]
1  Katie   38  [Sixes, Stache, Cisco]


|Function/Method|	Description|Use Case|
|---------------|--------------|--------|
`json.loads()`|	Converts JSON string to Python object	| Parsing JSON data from APIs or files
`json.dumps()`	|Converts Python object to JSON string	|Preparing data for API requests or storage
`pd.read_json()`	|Reads JSON file/string into DataFrame	|Loading JSON data directly into pandas
`df.to_json()`|	Converts DataFrame to JSON string|	Exporting data for web applications or APIs
orient parameter	| Controls structure of JSON output	|Formatting JSON for different requirements
`pd.json_normalize()`	|Flattens nested JSON structures	|Working with complex nested JSON data
JSON vs Python	| null→None, true→True, false→False |	Understanding data type conversions
Key Features	| Human-readable, language-independent	| Ideal for web APIs and configuration files


In [26]:
import lxml

### XML and HTML: Web Scraping

In [27]:
# Reading HTML tables automatically
tables = pd.read_html('examples/fdic_failed_bank_list.html', flavor='lxml')

# Check how many tables were found
print(f"Number of tables found: {len(tables)}")
# Output: Number of tables found: 1

# Get the first table
failures = tables[0]

# Show first few rows
print(failures.head())

Number of tables found: 4
                 Bank Name         City State Closing Date  \
0      First National Bank     New York    NY   2023-01-15   
1     Pacific Western Bank  Los Angeles    CA   2023-02-20   
2     Texas Community Bank      Houston    TX   2023-03-10   
3  Sunrise Bank of Florida        Miami    FL   2023-04-05   
4         Great Lakes Bank      Chicago    IL   2023-05-12   

   Assets (Millions $)               Status  
0                 1500               Closed  
1                 2300             Acquired  
2                 3200               Closed  
3                 4500  Under Investigation  
4                 2800               Closed  


In [28]:
# Convert date column to datetime format
close_timestamps = pd.to_datetime(failures['Closing Date'])

# Count bank failures by year
failures_by_year = close_timestamps.dt.year.value_counts()

print(failures_by_year)

Closing Date
2023    8
Name: count, dtype: int64


### Reading XML with lxml

In [29]:
# Import lxml library
from lxml import objectify

# Path to XML file
path = 'examples/Performance_MNR.xml'

### Parse the XML File

In [30]:
parsed = objectify.parse(path)
root = parsed.getroot()

print(root.tag)

INDICATORS


### Inspect the Root Element

In [31]:
print(root.tag)
print(len(root))

INDICATORS
1


### Extract Data from XML

In [32]:
data = []

skip_fields = {
    "PARENT_SEQ",
    "INDICATOR_SEQ",
    "DESIRED_CHANGE",
    "DECIMAL_PLACES",
}

### Extract Each Record

In [33]:
for indicator in root.INDICATOR:
    record = {}

    for child in indicator.getchildren():
        if child.tag in skip_fields:
            continue

        record[child.tag] = child.pyval

    data.append(record)

In [34]:
# Create a DataFrame
perf = pd.DataFrame(data)

perf.head()

,AGENCY_NAME,INDICATOR_NAME,DESCRIPTION,PERIOD_YEAR,PERIOD_MONTH,ACTUAL_VALUE,TARGET_VALUE,UNIT
0,New York City Transit,On-Time Performance,Percentage of trains arriving on time,2023,January,96.5,95.0,%
1,New York City Transit,Customer Satisfaction,Customer survey score,2023,February,89.2,90.0,%
2,New York City Transit,Service Reliability,Percentage of scheduled service delivered,2023,March,98.1,97.5,%


### Parsing HTML

In [35]:
from io import StringIO
from lxml import objectify

In [36]:
html = """
<a href="https://www.google.com">
    Google
</a>
"""

In [37]:
root = objectify.parse(
    StringIO(html)
).getroot()
print(root.get("href"))
print(root.text.strip())

https://www.google.com
Google


### **Key Differences Between HTML and XML**
|Feature|	HTML|	XML|
|-------|-------|------|
**Purpose**	| Displaying web pages |	Storing and transporting data
**Tags**|	Predefined (`div`, `p`, `table`)|	Custom (you can define your own)
**Flexibility**	|Limited	Highly flexible
**Errors**	|Browsers correct errors|	Must be perfectly correct (well-formed)

### **Summary Table of Functions and Methods**

|Method	| Library	| Purpose|	Sample Code|
|-------|-----------|--------|-------------|
Reading HTML Tables|	pandas	|Automatic table extraction	| `pd.read_html('file.html')`
Processing XML|	lxml	|Reading and writing XML	|`objectify.parse(open('file.xml'))`
Processing Complex HTML	| Beautiful Soup	|Messy or complex HTML	|`BeautifulSoup(html, 'html.parser')`
Date Conversion |	pandas	| Time series analysis	| `pd.to_datetime(column)`
Value Counts	| pandas	| Descriptive statistics |	`series.value_counts()`

### **Important Python Libraries for This Work**

| Library |	Purpose |	Advantages |
|---------|---------|--------------|
lxml|	Processing XML/HTML|	Fast and efficient
Beautiful Soup|	Processing HTML |	Handles messy HTML better
html5lib	| Processing HTML	| Most standards-compliant method